# Zika - Modelling

In [44]:
%reload_ext autoreload
%autoreload 2
from my_lib import *

In [45]:
z_features = ['EW', 'EW_start_date', 'pop', 'Month']
target = 'cases'

## Train Model on all data

In [46]:
from sklearn.linear_model import PoissonRegressor
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=['object', 'category']))
])

modelling_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', PoissonRegressor(max_iter=300))
])

In [47]:
from scipy.stats import loguniform
param_grid = {
    'model__alpha': loguniform(1e-4, 10.0),
    'model__max_iter': [100, 200, 300]
}

for location in "ABCDEF":
    print(f"Location {location}")
    df_z = pd.read_feather(f"data/zika_{location}.feather")
    df_d = pd.read_feather(f"data/dengue_{location}.feather")
    df_s = pd.read_feather(f"data/score_{location}.feather")
    df_w = pd.read_feather(f"data/weather_{location}.feather")

    X_train, X_test, y_train, y_test = prepare_mining_data(
        df_z,
        df_s,
        df_d,
        df_w,
        z_features,
        target
    )

    best_pipeline = tune_pipeline(X_train, y_train, modelling_pipeline, param_grid, 5)

    # modelling_pipeline.fit(X_train, y_train)
    y_pred = best_pipeline.predict(X_test)

    df_s[target] = y_pred
    y_pred = np.clip(y_pred, 0, None)

    df_s.to_csv(f"output/baseline_model_{location}.csv", index=False)

Location A
  -> Best Parameters: {'model__alpha': np.float64(9.582893421089915), 'model__max_iter': 100}
  -> Best Validation MAE: 3.6464
MAE across folds: [ 2.31566172  1.74988396 22.9731531   1.30280232  1.25409715]
Mean Validation MAE: 5.919119650433609
Location B
  -> Best Parameters: {'model__alpha': np.float64(6.0067294657128025), 'model__max_iter': 300}
  -> Best Validation MAE: 2.9435
MAE across folds: [12.76334081  2.37567296  3.3443291   0.90065361  3.39675708]
Mean Validation MAE: 4.5561507115528155
Location C
  -> Best Parameters: {'model__alpha': np.float64(5.007769186311567), 'model__max_iter': 300}
  -> Best Validation MAE: 1.8914
MAE across folds: [0.27227798 1.52878504 1.63347533 2.31892629 4.16974707]
Mean Validation MAE: 1.9846423410757374
Location D
  -> Best Parameters: {'model__alpha': np.float64(3.4800906519247947), 'model__max_iter': 200}
  -> Best Validation MAE: 4.4854
MAE across folds: [6.22050882 4.51481275 7.42641973 2.91597364 1.85335359]
Mean Validation M

In [48]:
target_mean = np.mean(y_train)
target_std = np.std(y_train)

print(f"Target Mean: {target_mean:.2f}")
print(f"Target Std Dev: {target_std:.2f}")
print(f"Relative Error: {(10 / target_mean) * 100:.1f}%")

Target Mean: 1.28
Target Std Dev: 3.02
Relative Error: 778.6%


## Build Submission File

In [ ]:
def make_sub_file(name="baseline_model"):
    dfs = [pd.read_csv(f"output/{name}_{location}.csv") for location in "ABCDEF"]
    df = pd.concat(dfs, ignore_index=True)
    df['ID'] = df['location'] + df['EW'].astype(str)
    df[['ID', 'cases']].to_csv(f"output/{name}_submission.csv", index=False)

make_sub_file()

FileNotFoundError: [Errno 2] No such file or directory: 'output/poisson_A.csv'